# 01 - Synthetic Data Generation
Notebook wrapper for src/generate_data.py.

This notebook generates all base Parquet tables in data/ with a reproducible seed.

In [ ]:
from pathlib import Path
import hashlib
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Using project root: {PROJECT_ROOT}')

In [ ]:
from src.generate_data import GenerationConfig, generate_dataset, save_dataset, summarize_outputs

cfg = GenerationConfig(
    seed=42,
    n_borrowers=90000,
    avg_group_size=12,
    target_default_rate=0.18,
    output_dir=Path('data'),
)

tables = generate_dataset(cfg)
save_dataset(tables, cfg.output_dir)
summarize_outputs(cfg.output_dir, tables['labels'])

In [ ]:
expected = [
    'borrowers.parquet',
    'loans.parquet',
    'repayments.parquet',
    'groups.parquet',
    'edges.parquet',
    'cdr.parquet',
    'mobile_events.parquet',
    'labels.parquet',
]

for name in expected:
    p = Path('data') / name
    print(f'{name:<24} exists={p.exists()} size_mb={p.stat().st_size / (1024 * 1024):.2f}' if p.exists() else f'{name:<24} exists=False')

print('\nObserved default rate:', tables['labels']['default_flag'].mean())

In [ ]:
def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

hashes = {}
for p in sorted(Path('data').glob('*.parquet')):
    hashes[p.name] = file_sha256(p)

hashes